# Player Selection Support System

Case Study 104 — complete ML workflow.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, ConfusionMatrixDisplay


In [ ]:
df = pd.read_csv('../data/player_selection_data.csv')
print('Shape:', df.shape)
print(df.isnull().sum())
print('Duplicate IDs:', df['player_id'].duplicated().sum())
df.head()

In [ ]:
df = df.drop_duplicates(subset='player_id').copy()
features = ['age','matches','innings','not_outs','runs','batting_average','strike_rate','centuries','fifties','ducks','recent_form_score','fitness_score','fielding_score']
X=df[features]; y=df['selected']
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.20,random_state=42,stratify=y)
print('Training rows:',len(X_train),'Testing rows:',len(X_test))

In [ ]:
plt.figure(figsize=(7,4)); sns.countplot(x=df['selected']); plt.title('Selected vs Not Selected'); plt.show()

In [ ]:
plt.figure(figsize=(7,5)); sns.scatterplot(data=df,x='batting_average',y='strike_rate',hue='selected'); plt.title('Batting Average vs Strike Rate'); plt.show()

In [ ]:
plt.figure(figsize=(10,7)); sns.heatmap(df[features+['selected']].corr(),center=0); plt.title('Correlation Heatmap'); plt.show()

In [ ]:
models={
'Logistic Regression':Pipeline([('imputer',SimpleImputer(strategy='median')),('scaler',StandardScaler()),('model',LogisticRegression(max_iter=2000,random_state=42))]),
'Decision Tree':Pipeline([('imputer',SimpleImputer(strategy='median')),('model',DecisionTreeClassifier(max_depth=5,min_samples_leaf=8,random_state=42))]),
'Random Forest':Pipeline([('imputer',SimpleImputer(strategy='median')),('model',RandomForestClassifier(n_estimators=300,max_depth=7,min_samples_leaf=4,random_state=42,class_weight='balanced'))])}
rows=[]
for name,m in models.items():
    m.fit(X_train,y_train); p=m.predict(X_test)
    rows.append([name,accuracy_score(y_test,p),precision_score(y_test,p),recall_score(y_test,p),f1_score(y_test,p)])
results=pd.DataFrame(rows,columns=['Model','Accuracy','Precision','Recall','F1'])
results.sort_values('F1',ascending=False)

In [ ]:
best_name=results.sort_values('F1',ascending=False).iloc[0]['Model']; best=models[best_name]; pred=best.predict(X_test)
print('Best model:',best_name)
print('Accuracy:',accuracy_score(y_test,pred)); print('Precision:',precision_score(y_test,pred)); print('Recall:',recall_score(y_test,pred)); print('F1:',f1_score(y_test,pred))
ConfusionMatrixDisplay.from_predictions(y_test,pred,display_labels=['Not Selected','Selected']); plt.title('Confusion Matrix'); plt.show()

## Conclusion
The final model is selected using the highest test-set F1-score. Missing numerical values are imputed using the median within the modelling pipeline, preventing data leakage from the test set. The system is a decision-support prototype; its target is an academic proxy label, not a verified selector decision.